# Popularity vs Quality (simple version)

## Do movies with more ratings have, on average, higher ratings?

- **Popularity** = number of ratings a movie received.
- **Quality** = the movie's average rating (MovieLens scale 0.5–5).

**Data:** MovieLens 100K (100,836 ratings, 610 users, 9,742 movies) and IMDb (`title.basics`,
`title.ratings`) as an outside check. The raw files are converted to **Parquet**, loaded as **DuckDB tables**.

**Index**

1. How is popularity spread across movies?
2. Main answer: average rating per popularity level
3. Checking the answer: (a) user bias, (b) release decade and genre, (c) IMDb
4. Top 20 popular vs Top 20 rated
5. Conclusions


## 0. Setup


In [1]:
import os
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
from IPython.display import display

# MovieLens version: "ml-100k" (ml-latest-small, default) or "ml-32m".
# Set it with the BDF_DATASET environment variable or change the default here.
DATASET = os.environ.get("BDF_DATASET", "ml-100k")

ROOT = Path.cwd().resolve()
DATA_DIR = next(
    (d for d in (ROOT / "data", ROOT.parent / "data") if d.is_dir()), None
)
if DATA_DIR is None:
    raise FileNotFoundError("data/ folder not found: see the README to download the data")
ML_DIR = DATA_DIR / DATASET  # MovieLens CSVs
IMDB_DIR = DATA_DIR / "imdb"  # IMDb TSVs
if not ML_DIR.is_dir():
    raise FileNotFoundError(f"{ML_DIR} not found: see the README to download {DATASET}")
PQ_DIR = ROOT / "parquet"  # parquet/<DATASET>/ for MovieLens, parquet/imdb/ for IMDb
DB_PATH = ROOT / f"popularity_vs_quality_{DATASET}.duckdb"
print(f"Dataset: {DATASET}")

MIN_RATINGS = (
    50  # a movie needs at least this many ratings for its average to be trusted
)

PRIMARY, ACCENT = "#2a78d6", "#eb6834"
px.defaults.template = "plotly_white"


def spearman(df, x="n_ratings", y="avg_rating"):
    return round(df.select(pl.corr(x, y, method="spearman")).item(), 3)

Dataset: ml-100k


## Load the data

Each raw file is converted to Parquet with one DuckDB `COPY`. IMDb is reduced to feature films while it is read. Then each Parquet file becomes a
table in `popularity_vs_quality_<dataset>.duckdb`.

The MovieLens version is set with `DATASET` (environment variable `BDF_DATASET`): `ml-100k` (default, used for the text of this notebook) or `ml-32m`. Parquet files go to `parquet/<dataset>/` and `parquet/imdb/`, and are rebuilt when the raw file is newer.


In [2]:
def p(path):
    return path.resolve().as_posix()


SOURCES = {
    "movies": f"SELECT * FROM read_csv_auto('{p(ML_DIR / 'movies.csv')}')",
    "ratings": f"SELECT * FROM read_csv_auto('{p(ML_DIR / 'ratings.csv')}')",
    "links": f"SELECT * FROM read_csv('{p(ML_DIR / 'links.csv')}', types={{'imdbId': 'VARCHAR'}})",
    "imdb_basics": f"""
        SELECT tconst, primaryTitle, TRY_CAST(startYear AS INTEGER) AS startYear, genres
        FROM read_csv('{p(IMDB_DIR / 'title.basics.tsv')}', delim='\t', quote='', header=true,
                      nullstr='\\N', all_varchar=true)
        WHERE titleType = 'movie'""",
    "imdb_ratings": f"""
        SELECT tconst, CAST(averageRating AS DOUBLE) AS averageRating, CAST(numVotes AS INTEGER) AS numVotes
        FROM read_csv('{p(IMDB_DIR / 'title.ratings.tsv')}', delim='\t', quote='', header=true,
                      nullstr='\\N', all_varchar=true)""",
}

RAW_FILES = {  # the file each table is converted from
    "movies": ML_DIR / "movies.csv",
    "ratings": ML_DIR / "ratings.csv",
    "links": ML_DIR / "links.csv",
    "imdb_basics": IMDB_DIR / "title.basics.tsv",
    "imdb_ratings": IMDB_DIR / "title.ratings.tsv",
}

con = duckdb.connect(str(DB_PATH))
for name, select in SOURCES.items():
    folder = PQ_DIR / ("imdb" if name.startswith("imdb_") else DATASET)
    folder.mkdir(parents=True, exist_ok=True)
    parquet = folder / f"{name}.parquet"
    # (re)build when missing or older than its raw file, so a new download is never ignored
    if not parquet.exists() or parquet.stat().st_mtime < RAW_FILES[name].stat().st_mtime:
        con.execute(f"COPY ({select}) TO '{p(parquet)}' (FORMAT PARQUET)")
    con.execute(
        f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_parquet('{p(parquet)}')"
    )


def q(sql):
    """Run a query, show the result and return it as a DataFrame."""
    df = con.sql(sql).pl()
    display(df)
    return df


_ = q("""
SELECT 'movies' AS table_name, COUNT(*) AS rows FROM movies
UNION ALL SELECT 'ratings', COUNT(*) FROM ratings
UNION ALL SELECT 'links', COUNT(*) FROM links
UNION ALL SELECT 'imdb_basics', COUNT(*) FROM imdb_basics
UNION ALL SELECT 'imdb_ratings', COUNT(*) FROM imdb_ratings
""")

table_name,rows
str,i64
"""movies""",9742
"""ratings""",100836
"""links""",9742
"""imdb_basics""",761268
"""imdb_ratings""",1733543


### One row per movie: `movie_stats`

Almost every question starts from the same per-movie numbers, so we compute them once and keep them as a table:
release year, genres, number of ratings, average rating, and a **popularity bucket**
(1 rating, 2–4, 5–9, 10–19, 20–49, 50–99, 100+). Movies nobody rated have no average and are left out.


In [3]:
con.execute(r"""
CREATE OR REPLACE TABLE movie_stats AS
SELECT m.movieId,
       m.title,
       m.genres,
       TRY_CAST(regexp_extract(m.title, '\((\d{4})\)\s*$', 1) AS INTEGER) AS year,
       COUNT(*)      AS n_ratings,
       AVG(r.rating) AS avg_rating,
       CASE WHEN COUNT(*) = 1   THEN 1 WHEN COUNT(*) < 5   THEN 2 WHEN COUNT(*) < 10 THEN 3
            WHEN COUNT(*) < 20  THEN 4 WHEN COUNT(*) < 50  THEN 5 WHEN COUNT(*) < 100 THEN 6
            ELSE 7 END AS bucket_id,
       CASE WHEN COUNT(*) = 1   THEN '1'     WHEN COUNT(*) < 5   THEN '2-4'   WHEN COUNT(*) < 10 THEN '5-9'
            WHEN COUNT(*) < 20  THEN '10-19' WHEN COUNT(*) < 50  THEN '20-49' WHEN COUNT(*) < 100 THEN '50-99'
            ELSE '100+' END AS bucket
FROM movies m
JOIN ratings r ON r.movieId = m.movieId
GROUP BY m.movieId, m.title, m.genres
""")

_ = q("SELECT * FROM movie_stats ORDER BY n_ratings DESC LIMIT 5")

movieId,title,genres,year,n_ratings,avg_rating,bucket_id,bucket
i64,str,str,i32,i64,f64,i32,str
356,"""Forrest Gump (1994)""","""Comedy|Drama|Romance|War""",1994,329,4.164134,7,"""100+"""
318,"""Shawshank Redemption, The (199…","""Crime|Drama""",1994,317,4.429022,7,"""100+"""
296,"""Pulp Fiction (1994)""","""Comedy|Crime|Drama|Thriller""",1994,307,4.197068,7,"""100+"""
593,"""Silence of the Lambs, The (199…","""Crime|Horror|Thriller""",1991,279,4.16129,7,"""100+"""
2571,"""Matrix, The (1999)""","""Action|Sci-Fi|Thriller""",1999,278,4.192446,7,"""100+"""


## 1. How is popularity spread across movies?


In [4]:
spread = q("""
SELECT bucket,
       COUNT(*) AS movies,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM movie_stats), 1) AS percentage_movies,
       ROUND(100.0 * SUM(n_ratings) / (SELECT COUNT(*) FROM ratings), 1) AS percentage_ratings
FROM movie_stats
GROUP BY bucket_id, bucket
ORDER BY bucket_id
""")

fig = px.bar(
    spread,
    x="bucket",
    y=["percentage_movies", "percentage_ratings"],
    barmode="group",
    color_discrete_sequence=["#c9c8c2", PRIMARY],
    labels={"bucket": "Ratings per movie", "value": "%", "variable": ""},
    title="Most movies are rarely rated; most ratings go to a few movies",
)
fig.show()

bucket,movies,percentage_movies,percentage_ratings
str,i64,f64,f64
"""1""",3446,35.4,3.4
"""2-4""",2628,27.0,7.1
"""5-9""",1381,14.2,9.1
"""10-19""",972,10.0,13.1
"""20-49""",847,8.7,26.3
"""50-99""",312,3.2,21.0
"""100+""",138,1.4,20.0


### Conclusion 1

- **35% of the movies have a single rating**, and 62% have fewer than 5. Together they get only 10.5% of all ratings.
- The **450 movies with ≥ 50 ratings** (4.6% of the catalogue) get **41% of all ratings**.
- So the typical movie in the catalogue is one almost nobody rated.


## 2. Main answer: average rating per popularity level

For each popularity bucket, we calculated the average of the movies' average ratings (each movie counts once).


In [5]:
by_bucket = q("""
SELECT bucket,
       COUNT(*) AS movies,
       ROUND(AVG(avg_rating), 2) AS mean_rating,
       ROUND(MEDIAN(avg_rating), 2) AS median_rating
FROM movie_stats
GROUP BY bucket_id, bucket
ORDER BY bucket_id
""")

fig = px.line(
    by_bucket,
    x="bucket",
    y="mean_rating",
    markers=True,
    labels={
        "bucket": "Ratings per movie (popularity)",
        "mean_rating": "Average rating",
    },
    title="Average rating per popularity level",
)
fig.update_traces(line_color=PRIMARY)
fig.show()

bucket,movies,mean_rating,median_rating
str,i64,f64,f64
"""1""",3446,3.23,3.5
"""2-4""",2628,3.18,3.25
"""5-9""",1381,3.21,3.25
"""10-19""",972,3.31,3.36
"""20-49""",847,3.45,3.49
"""50-99""",312,3.59,3.66
"""100+""",138,3.82,3.88


The plot shows that, apparently, the more popular movies (according to the number of ratings) tend to be better rated and more acclaimed. The same idea in one number: the Spearman correlation between number of ratings and average rating should show the effect of movie popularity on its rating. We compute it for all movies and then only for movies with at least 10, 50 and 100 ratings.


In [6]:
movies_df = con.sql("SELECT n_ratings, avg_rating FROM movie_stats").pl()

pl.DataFrame(
    [
        {
            "min_ratings": n,
            "movies": movies_df.filter(pl.col("n_ratings") >= n).height,
            "spearman": spearman(movies_df.filter(pl.col("n_ratings") >= n)),
        }
        for n in [1, 10, MIN_RATINGS, 100]
    ]
)

min_ratings,movies,spearman
i64,i64,f64
1,9724,0.04
10,2269,0.251
50,450,0.311
100,138,0.32


It makes sense that the correlation is so low (**0.04**) for all movies. An average of 1 or 2 ratings can be anything from 0.5 to 5.
The following table shows, per bucket, how spread out the averages are and how many are extreme (≥ 4.5 or ≤ 1.5). Additionally, it becomes clear that the standard deviation of the average rating decreases as we limit the dataset to more popular movies.


In [7]:
_ = q("""
SELECT bucket,
       ROUND(STDDEV(avg_rating), 2) AS spread_of_averages,
       ROUND(100.0 * AVG(CASE WHEN avg_rating >= 4.5 OR avg_rating <= 1.5 THEN 1 ELSE 0 END), 1) AS percentage_extreme
FROM movie_stats
GROUP BY bucket_id, bucket
ORDER BY bucket_id
""")

bucket,spread_of_averages,percentage_extreme
str,f64,f64
"""1""",1.13,28.1
"""2-4""",0.82,9.0
"""5-9""",0.65,1.6
"""10-19""",0.56,0.4
"""20-49""",0.45,0.0
"""50-99""",0.38,0.0
"""100+""",0.34,0.0


### Conclusion 2

- **Up to ~10 ratings the average is flat at about 3.2.** Then it rises with every bucket: 3.31 (10–19), 3.45
  (20–49), 3.59 (50–99) and **3.82 (100+)**. The gap between 2–4 and 100+ ratings is **0.64 stars**.
- **Over all movies the correlation is only 0.04**: 62% of the movies have fewer than 5 ratings and their averages
  are mostly noise (28.1% of single-rating movies have an extreme average; from 20 ratings up, none do).
- Once rarely rated movies are left out, it is clearly positive: **0.25** (>= 10), **0.31** (>= `MIN_RATINGS`), **0.32** (>= 100).
- Noise makes the small buckets _wider_, not _lower_: their median is also lower (3.25 vs 3.88 for 100+). In other words, less popular movies tend to have more extreme ratings.
- From here on we use movies with enough ratings (>= 50 or >= 10 when we split into groups).


## 3. Checking the answer

This pattern shows apparent correlation, but the Spearman correlation value is still relatively low (around **0.3**). The next analysis will try to test the main conclusion from three different angles, to check its validity:

1. **User bias**: the effect of the type of rater in the film rating.
2. The effect of the movie's **genre** and **release decade**.
3. Does the conclusion hold against a different dataset (**IMDb**)?

### 3a. Is it _who_ rates them? (user bias)

Some users rate everything high, others everything low. If the users who rate many movies are harsher, and they are
the ones who find the obscure movies, those movies would look worse because of their audience, not because of the
movie.

With this in mind, we will summarize each user in a `user_stats` table, and users are split into 4 equal groups (quartiles) by how many movies they rated.


In [8]:
con.execute("""
CREATE OR REPLACE TABLE user_stats AS
SELECT userId, COUNT(*) AS n_ratings, AVG(rating) AS avg_rating
FROM ratings
GROUP BY userId
""")

_ = q("""
SELECT activity_quartile,
       COUNT(*) AS users,
       MIN(n_ratings) AS min_ratings,
       MAX(n_ratings) AS max_ratings,
       ROUND(AVG(avg_rating), 2) AS mean_rating,
       ROUND(MEDIAN(avg_rating), 2) AS median_rating,
       ROUND(100.0 * SUM(n_ratings) / (SELECT COUNT(*) FROM ratings), 1) AS percentage_of_all_ratings
FROM (SELECT *, NTILE(4) OVER (ORDER BY n_ratings) AS activity_quartile
      FROM user_stats)
GROUP BY activity_quartile
ORDER BY activity_quartile
""")

users_df = con.sql("SELECT userId, n_ratings, avg_rating FROM user_stats").pl()
print("Spearman, user activity vs user average rating:", spearman(users_df))

fig = px.scatter(
    users_df,
    x="n_ratings",
    y="avg_rating",
    log_x=True,
    opacity=0.6,
    labels={
        "n_ratings": "Ratings given by the user (log scale)",
        "avg_rating": "User's average rating",
    },
    title="User average rating, based on user activity",
)
fig.update_traces(marker_color=PRIMARY)
fig.show()

activity_quartile,users,min_ratings,max_ratings,mean_rating,median_rating,percentage_of_all_ratings
i64,i64,i64,i64,f64,f64,f64
1,153,20,35,3.68,3.77,4.0
2,153,35,71,3.73,3.78,7.7
3,152,72,168,3.68,3.72,17.1
4,152,168,2698,3.54,3.56,71.2


Spearman, user activity vs user average rating: -0.135


#### Heavy raters are harsher

- The correlation between how many movies a user rated and their average rating is **negative and weak** (−0.135).
- The difference comes from the **most active quartile** (168+ ratings each): they average **3.54**, against 3.68–3.73 for the other three quartiles. This is probably due to users who rate little tending to only rate the movies they liked; on the other hand, users who rate a lot also record the average and bad ones.
- The most active quartile gives **71% of all ratings**, heavily influencing the averages.

This shows that _who_ rates a movie matters. To remove this effect, we compare each rating with **that user's own average**: `rating - user average`. A movie with a positive value was rated above what its raters usually give.


In [9]:
con.execute("""
CREATE OR REPLACE TABLE movie_vs_user AS
SELECT r.movieId,
       AVG(r.rating - u.avg_rating) AS vs_user_avg,
       AVG(u.n_ratings) AS rater_activity
FROM ratings r
JOIN user_stats u ON u.userId = r.userId
GROUP BY r.movieId
""")

user_bias = q("""
SELECT s.bucket,
       COUNT(*) AS movies,
       ROUND(AVG(v.rater_activity)) AS avg_rater_activity,
       ROUND(AVG(s.avg_rating), 2)  AS mean_rating,
       ROUND(AVG(v.vs_user_avg), 2) AS mean_vs_user_avg
FROM movie_stats s
JOIN movie_vs_user v ON v.movieId = s.movieId
GROUP BY s.bucket_id, s.bucket
ORDER BY s.bucket_id
""")

fig = px.line(
    user_bias,
    x="bucket",
    y="mean_vs_user_avg",
    markers=True,
    labels={
        "bucket": "Ratings per movie (popularity)",
        "mean_vs_user_avg": "Rating − user average",
    },
    title="Average rating per popularity level, after removing user bias",
)
fig.update_traces(line_color=PRIMARY)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

popular = con.sql(f"""
SELECT s.n_ratings, v.vs_user_avg
FROM movie_stats s JOIN movie_vs_user v ON v.movieId = s.movieId
WHERE s.n_ratings >= {MIN_RATINGS}
""").pl()
print(
    f"Spearman (>= {MIN_RATINGS} ratings), after removing user bias:",
    spearman(popular, y="vs_user_avg"),
)

bucket,movies,avg_rater_activity,mean_rating,mean_vs_user_avg
str,i64,f64,f64,f64
"""1""",3446,1078.0,3.23,-0.14
"""2-4""",2628,999.0,3.18,-0.21
"""5-9""",1381,848.0,3.21,-0.19
"""10-19""",972,735.0,3.31,-0.14
"""20-49""",847,610.0,3.45,-0.05
"""50-99""",312,480.0,3.59,0.05
"""100+""",138,357.0,3.82,0.23


Spearman (>= 50 ratings), after removing user bias: 0.281


#### Conclusion 3a

- **The audience is different.** Raters of movies with only one rating have on average ~1,078 ratings each; raters of
  movies with more than 100 ratings have ~357. Heavy users are harsher, so part of the gap comes from _who_ rates.
- But **most of the gap between popular and non-popular movies remains.** It goes from 0.64 to **0.44 stars** (0.23 - (-0,21)), so **about two thirds of the
  gap is not user bias**. Movies with 100+ ratings get 0.23 stars more than their raters usually give.
- For movies with ≥ 50 ratings the correlation only drops from 0.31 to **0.28**.


### 3b. Is it release decade or genre?

Two other things could create the pattern:

- **Genre.** If some genres are rated higher _and_ are also the most watched, genre alone would make popular movies
  look better.
- **Release decade.** Old movies still watched today are the classics. If they are rated higher _and_ are also the
  popular ones, the pattern would come from age.

For each genre we count movies and ratings, the ratings per movie (popularity), the average rating, the
average compared with each rater's own average (`user_stats` from 3a), and the share of high (≥ 4.5) and low (≤ 2)
ratings. A movie with several genres counts in each of them.


In [10]:
genre_stats = q("""
SELECT g.genre,
       COUNT(DISTINCT r.movieId) AS movies,
       COUNT(*) AS ratings,
       ROUND(COUNT(*) / COUNT(DISTINCT r.movieId), 1) AS ratings_per_movie,
       ROUND(AVG(r.rating), 2) AS avg_rating,
       ROUND(AVG(r.rating - u.avg_rating), 2) AS vs_user_avg,
       ROUND(100.0 * AVG(CASE WHEN r.rating >= 4.5 THEN 1 ELSE 0 END), 1) AS percentage_high,
       ROUND(100.0 * AVG(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END), 1) AS percentage_low
FROM ratings r
JOIN movies m ON m.movieId = r.movieId
JOIN user_stats u ON u.userId = r.userId,
     UNNEST(STRING_SPLIT(m.genres, '|')) AS g(genre)
WHERE g.genre <> '(no genres listed)'
GROUP BY g.genre
ORDER BY avg_rating DESC
""")

fig = px.scatter(
    genre_stats,
    x="ratings_per_movie",
    y="avg_rating",
    text="genre",
    labels={
        "ratings_per_movie": "Ratings per movie (popularity)",
        "avg_rating": "Average rating",
    },
    title="Rating by genre, according to movie popularity",
)
fig.update_traces(marker_color=PRIMARY, textposition="top center")
fig.show()

genre,movies,ratings,ratings_per_movie,avg_rating,vs_user_avg,percentage_high,percentage_low
str,i64,i64,f64,f64,f64,f64,f64
"""Film-Noir""",85,870,10.2,3.92,0.35,34.0,5.9
"""War""",381,4859,12.8,3.81,0.24,32.0,7.8
"""Documentary""",438,1219,2.8,3.8,0.28,25.8,4.7
"""Crime""",1196,16681,13.9,3.66,0.13,26.7,9.9
"""Drama""",4349,41928,9.6,3.66,0.13,25.2,9.6
…,…,…,…,…,…,…,…
"""Sci-Fi""",980,17243,17.6,3.46,-0.06,21.2,15.2
"""Action""",1828,30635,16.8,3.45,-0.07,20.3,14.6
"""Children""",664,9208,13.9,3.41,-0.07,18.6,14.7


In [11]:
decade_stats = q("""
SELECT (s.year // 10) * 10 AS decade,
       COUNT(DISTINCT s.movieId) AS movies,
       ROUND(COUNT(*) / COUNT(DISTINCT s.movieId), 1) AS ratings_per_movie,
       ROUND(AVG(r.rating), 2) AS avg_rating
FROM movie_stats s
JOIN ratings r ON r.movieId = s.movieId
WHERE s.year >= 1950
GROUP BY decade
ORDER BY decade
""")

decade,movies,ratings_per_movie,avg_rating
i32,i64,f64,f64
1950,277,6.4,3.85
1960,399,7.2,3.81
1970,498,10.0,3.78
1980,1175,11.0,3.52
1990,2209,16.8,3.43
2000,2848,10.5,3.47
2010,1930,4.9,3.49


#### Genres and decades are rated differently

- **Genres differ by a little over half a star**: from **3.92** (Film-Noir), 3.81 (War) and 3.80 (Documentary) down to
  3.38 (Comedy) and **3.26** (Horror). Horror also has the most low ratings (21% are ≤ 2).
- **It is not user bias.** Compared with each rater's own average, the order barely changes: Film-Noir, Documentary
  and War stay on top (+0.24 to +0.35), Horror stays last (−0.20).
- **Older movies are rated higher**: 3.85 for the 1950s and 3.78 for the 1970s, against ~3.45 from the 1990s on.
- **But the high-rated groups are not the popular ones.** Documentary (2.8 ratings per movie), Film-Noir (10.2) and
  Drama (9.6) are among the _least_ watched; apart from IMAX (26.2, a small genre), the most watched are Adventure (19.1),
  Sci-Fi (17.6) and Action (16.8), all rated around or below the overall average (3.50). Old decades also have fewer ratings per movie (6–10) than the 1990s (16.8).
- So genre and decade _do_ change the ratings, but they push in the **opposite** direction to our pattern. To be
  sure, we now look at the correlation **inside** each decade and each genre. We use movies with ≥ 10 ratings, so
  that there are enough movies in each group.


In [12]:
by_decade = con.sql("""
SELECT (year // 10) * 10 AS decade, n_ratings, avg_rating
FROM movie_stats
WHERE n_ratings >= 10 AND year >= 1950
""").pl()

decade_corr = (
    by_decade.group_by("decade")
    .agg(
        movies=pl.len(),
        spearman=pl.corr("n_ratings", "avg_rating", method="spearman").round(3),
    )
    .sort("decade")
    .with_columns(decade_label=pl.format("{}s", "decade"))
)
display(decade_corr)

fig = px.bar(
    decade_corr,
    x="decade_label",
    y="spearman",
    labels={"decade_label": "Release decade", "spearman": "Spearman (popularity vs rating)"},
    title="The relationship holds inside every decade",
)
fig.update_traces(marker_color=PRIMARY)
fig.show()

decade,movies,spearman,decade_label
i32,u32,f64,str
1950,45,0.17,"""1950s"""
1960,70,0.38,"""1960s"""
1970,100,0.343,"""1970s"""
1980,301,0.342,"""1980s"""
1990,758,0.254,"""1990s"""
2000,707,0.342,"""2000s"""
2010,247,0.413,"""2010s"""


In [13]:
by_genre = con.sql("""
SELECT g.genre, s.n_ratings, s.avg_rating
FROM movie_stats s,
     UNNEST(STRING_SPLIT(s.genres, '|')) AS g(genre)
WHERE s.n_ratings >= 10 AND g.genre <> '(no genres listed)'
""").pl()

genre_corr = (
    by_genre.group_by("genre")
    .agg(
        movies=pl.len(),
        spearman=pl.corr("n_ratings", "avg_rating", method="spearman").round(3),
    )
    .filter(pl.col("movies") >= MIN_RATINGS)  # skip genres with too few movies
    .sort("spearman")
)
display(genre_corr)

fig = px.bar(
    genre_corr,
    x="spearman",
    y="genre",
    orientation="h",
    height=550,
    labels={"spearman": "Spearman (popularity vs rating)", "genre": ""},
    title="…and inside nearly every genre",
)
fig.update_traces(marker_color=PRIMARY)
fig.show()

genre,movies,spearman
str,u32,f64
"""Animation""",155,0.185
"""Drama""",963,0.189
"""Musical""",92,0.223
"""Comedy""",911,0.265
"""War""",104,0.266
…,…,…
"""Adventure""",500,0.349
"""Fantasy""",261,0.374
"""Children""",212,0.407


#### Conclusion 3b

- **Positive in every decade from the 1950s to the 2010s** (0.17 to 0.41). It is not just old classics.
- **Positive in all genres** with enough movies (from ~0.19 for Drama and Animation to 0.42 for Horror).
- Inside each group, the more popular movies are still rated higher, so neither decade nor genre creates the pattern.


### 3c. Does a different dataset corroborate? (IMDb)

In MovieLens, popularity and quality come from the **same ratings**. IMDb gives an independent audience.
`links.imdbId` plus the prefix `tt` is IMDb's `tconst`.

First: the **same movies**, grouped by MovieLens popularity, with their MovieLens average (× 2, to put it on IMDb's
1–10 scale) next to their IMDb rating.


In [14]:
ml_imdb = q("""
SELECT s.bucket,
       COUNT(*) AS movies,
       ROUND(AVG(s.avg_rating) * 2, 2) AS movielens_x2,
       ROUND(AVG(i.averageRating), 2)  AS imdb_rating
FROM movie_stats s
JOIN links l ON l.movieId = s.movieId
JOIN imdb_ratings i ON i.tconst = 'tt' || l.imdbId
GROUP BY s.bucket_id, s.bucket
ORDER BY s.bucket_id
""")

fig = px.line(
    ml_imdb,
    x="bucket",
    y=["movielens_x2", "imdb_rating"],
    markers=True,
    color_discrete_sequence=[PRIMARY, ACCENT],
    labels={
        "bucket": "MovieLens ratings per movie (popularity)",
        "value": "Average rating (1–10)",
        "variable": "",
    },
    title="IMDb voters see the same pattern on the same movies",
)
fig.show()

same_movies = con.sql("""
SELECT s.n_ratings, i.averageRating AS imdb_rating
FROM movie_stats s
JOIN links l ON l.movieId = s.movieId
JOIN imdb_ratings i ON i.tconst = 'tt' || l.imdbId
WHERE s.n_ratings >= 10
""").pl()
print(
    "Spearman, MovieLens popularity vs IMDb rating (>= 10 ratings):",
    spearman(same_movies, y="imdb_rating"),
)

bucket,movies,movielens_x2,imdb_rating
str,i64,f64,f64
"""1""",3431,6.46,6.48
"""2-4""",2623,6.36,6.53
"""5-9""",1380,6.43,6.61
"""10-19""",971,6.63,6.76
"""20-49""",846,6.89,7.02
"""50-99""",312,7.19,7.4
"""100+""",138,7.64,7.9


Spearman, MovieLens popularity vs IMDb rating (>= 10 ratings): 0.354


The pattern apparently holds for the same movies on the IMDb dataset: the average rating of the more popular ones tends to be higher than the more obscure. Additionally, performing the same analysis on **all IMDb feature films** with at least 100 votes results in the same conclusion, as shown below.


In [15]:
all_imdb = q("""
SELECT CASE WHEN r.numVotes < 1000    THEN '100-999'
            WHEN r.numVotes < 10000   THEN '1k-9.9k'
            WHEN r.numVotes < 100000  THEN '10k-99k'
            WHEN r.numVotes < 1000000 THEN '100k-999k'
            ELSE '1M+' END AS votes,
       COUNT(*) AS movies,
       ROUND(AVG(r.averageRating), 2) AS mean_rating
FROM imdb_basics b
JOIN imdb_ratings r ON r.tconst = b.tconst
WHERE r.numVotes >= 100
GROUP BY votes
ORDER BY MIN(r.numVotes)
""")

fig = px.line(
    all_imdb,
    x="votes",
    y="mean_rating",
    markers=True,
    labels={
        "votes": "IMDb votes per movie",
        "mean_rating": "Average IMDb rating (1–10)",
    },
    title="On all of IMDb, the most-voted films are also rated higher",
)
fig.update_traces(line_color=ACCENT)
fig.show()

imdb_df = con.sql("""
SELECT r.numVotes, r.averageRating
FROM imdb_basics b JOIN imdb_ratings r ON r.tconst = b.tconst
WHERE r.numVotes >= 100
""").pl()
print(
    f"Spearman, all IMDb films ({len(imdb_df):,}):",
    spearman(imdb_df, x="numVotes", y="averageRating"),
)

votes,movies,mean_rating
str,i64,f64
"""100-999""",98717,5.74
"""1k-9.9k""",36850,6.11
"""10k-99k""",9933,6.47
"""100k-999k""",2680,6.94
"""1M+""",90,8.37


Spearman, all IMDb films (148,270): 0.174


#### Conclusion 3c

- **Same movies, other audience: the relationship is even stronger.** MovieLens popularity vs IMDb rating gives
  **0.35**. The IMDb rating climbs from ~6.5 to **7.9** with MovieLens
  popularity, following the same curve.
- **All of IMDb (~148,000 films): 0.174**, and the average rating climbs with every level of votes: 5.74 (100–999)
  to **8.37** (1M+ votes).
- In other words, the pattern is not specific to MovieLens.


## 4. Top 20 popular vs Top 20 rated

It is also useful to understand the overlap between the most popular and the best rated movies. Among movies with ≥ 50 ratings, we will check which ones are in **both** the top 20 by
number of ratings and the top 20 by average rating.


In [16]:
both = q(f"""
SELECT title, n_ratings, ROUND(avg_rating, 2) AS avg_rating
FROM movie_stats
WHERE movieId IN (SELECT movieId FROM movie_stats WHERE n_ratings >= {MIN_RATINGS}
                  ORDER BY n_ratings DESC LIMIT 20)
  AND movieId IN (SELECT movieId FROM movie_stats WHERE n_ratings >= {MIN_RATINGS}
                  ORDER BY avg_rating DESC LIMIT 20)
ORDER BY avg_rating DESC
""")
print("Movies in both Top 20 lists:", len(both))

_ = q(f"""
SELECT 'Top 20 most popular' AS movie_group, ROUND(AVG(avg_rating), 2) AS mean_rating
FROM (SELECT avg_rating FROM movie_stats ORDER BY n_ratings DESC LIMIT 20)
UNION ALL
SELECT 'All with >= {MIN_RATINGS} ratings', ROUND(AVG(avg_rating), 2) FROM movie_stats WHERE n_ratings >= {MIN_RATINGS}
UNION ALL
SELECT 'All movies', ROUND(AVG(avg_rating), 2) FROM movie_stats
""")

title,n_ratings,avg_rating
str,i64,f64
"""Shawshank Redemption, The (199…",317,4.43
"""Fight Club (1999)""",218,4.27
"""Usual Suspects, The (1995)""",204,4.24
"""Star Wars: Episode IV - A New …",251,4.23
"""Schindler's List (1993)""",220,4.22
"""Star Wars: Episode V - The Emp…",211,4.22
"""Raiders of the Lost Ark (India…",200,4.21


Movies in both Top 20 lists: 7


movie_group,mean_rating
str,f64
"""Top 20 most popular""",4.08
"""All with >= 50 ratings""",3.66
"""All movies""",3.26


### Conclusion 4

- **7 movies are in both Top 20 lists**: _Shawshank Redemption_,
  _Fight Club_, _The Usual Suspects_, the first two _Star Wars_, _Schindler's List_ and _Raiders of the Lost Ark_.
- The **Top 20 most popular average 4.08**, against 3.66 for all movies with ≥ 50 ratings and 3.26 for all movies.
- The most watched movies are nearly all well rated, but 13 of the 20 best-rated movies are not among the most
  watched. The link is there, but it is moderate.


In [17]:
con.close()

## 5. Conclusions

**Do movies with more ratings have, on average, higher ratings? Yes, but the relationship is moderate, and it only
shows up once rarely rated movies are left out.**

1. **Yes, on average.** Movies with 100+ ratings average **3.82**, against ~**3.2** for movies with fewer than 10.
   Among movies with ≥ 50 ratings, the Spearman correlation is **0.31**.
2. **Over the whole catalogue it is almost invisible** (0.04). Most movies have 1 to 4 ratings and their averages
   are mostly noise, so they must be left out.
3. **It survives the checks:**
   - removing user bias (about two thirds of the gap remains);
   - looking inside each release decade (positive in all) and each genre (positive in all);
   - an independent audience: the IMDb rating of the MovieLens movies (0.35) and all of IMDb (0.17).
4. **But it is moderate.** A correlation of ~0.3 leaves most of the variation unexplained: many popular movies are
   average, many excellent movies are little seen, and only 7 of the Top 20 popular are in the Top 20 rated.

| Check                                                 | Result                                |
| ----------------------------------------------------- | ------------------------------------- |
| Average rating, 2–4 ratings vs 100+ ratings           | 3.18 vs 3.82 (+0.64)                  |
| Spearman, all movies / >= 10 / >= 50 / >= 100 ratings | 0.04 / 0.25 / 0.31 / 0.32             |
| Gap after removing user bias                          | +0.43 (about two thirds of the gap)   |
| Spearman inside release decades (1950s–2010s)         | 0.17 to 0.41, all positive            |
| Spearman inside genres (≥ 50 movies)                  | positive in all   |
| Top 20 popular and Top 20 rated (≥ 50 ratings)        | 7 movies belong to both Top 20 lists  |
| IMDb rating vs MovieLens popularity (≥ 10 ratings)    | 0.35                                  |
| All IMDb films (>= 100 votes)                         | 0.17; 5.74 to 8.37 across popularity levels |

Overall, we believe that this relationship **is an association, not a cause.** Good movies attract viewers, be it through charts or word of mouth: the more people watch a movie, the more people talk about it. This generates more viewers and the cycle repeats. Moreover, people choose movies they expect to like: ratings tend to be higher for more "casual" viewers, due to this expectation.

In the end, our belief is that Popularity is a **signal** of movie quality, but not a guarantee.
